# Fase 1: Data Infestion & Cleaning

In [ ]:
# --- FASE 1: Data Ingestion & Cleaning ---
import pandas as pd
import numpy as np

# Percorsi dei file
file_path_storico = "./dataset/Archivio con lo storico prezzi dei titoli in formato CSV(Ultimo aggiornamento 07-05-2026).csv"
file_path_anagrafica = "./dataset/Obbligazioni quotate su Borsa Italiana(Ultimo aggiornamento 07-05-2026).csv"

# Caricamento storico prezzi 
df_storico = pd.read_csv(file_path_storico,
                        sep=';',
                        decimal=',',
                        parse_dates=['referencedate', 'endvaluedate'],
                        dayfirst=True)

# Caricamento anagrafica obbligazioni
try:
    df_anagrafica = pd.read_csv(file_path_anagrafica, sep=';', decimal=',')
    print(f"Anagrafica caricata: {df_anagrafica.shape[0]} righe")
except Exception as e:
    print("Errore nel caricamento dell'anagrafica:", e)
    df_anagrafica = None

# Pulizia base: rimozione spazi, uniformità nomi colonne
if df_anagrafica is not None:
    df_anagrafica.columns = df_anagrafica.columns.str.strip().str.lower()
    df_storico.columns = df_storico.columns.str.strip().str.lower()

# Cast di alcune colonne (esempio: rating a stringa, isin a stringa)
if df_anagrafica is not None:
    if 'rating' in df_anagrafica.columns:
        df_anagrafica['rating'] = df_anagrafica['rating'].astype(str)
    if 'isincode' in df_anagrafica.columns:
        df_anagrafica['isincode'] = df_anagrafica['isincode'].astype(str)
    if 'isincode' in df_storico.columns:
        df_storico['isincode'] = df_storico['isincode'].astype(str)

# Visualizzazione di controllo
print("\nDimensioni storico prezzi:", df_storico.shape)
if df_anagrafica is not None:
    print("Dimensioni anagrafica:", df_anagrafica.shape)
    display(df_anagrafica.head())
display(df_storico.head())

In [ ]:
# Estraiamo gli ISIN unici sotto forma di "Insiemi" (Set)
isins_in_anagrafica = set(df_anagrafica['isincode'].unique())
isins_in_storico = set(df_storico['isincode'].unique())

# Analisi dell'intersezione (Il KDD process)
common_isins = isins_in_anagrafica.intersection(isins_in_storico)
only_in_anagrafica = isins_in_anagrafica - isins_in_storico
only_in_storico = isins_in_storico - isins_in_anagrafica

# Stampiamo il report
print(f"--- REPORT DATA CONSISTENCY ---")
print(f"Totale ISIN unici in Anagrafica: {len(isins_in_anagrafica)}")
print(f"Totale ISIN unici in Storico:    {len(isins_in_storico)}")
print(f"ISIN COMUNI (Utilizzabili per ML): {len(common_isins)}")
print(f"-------------------------------")
print(f"ISIN scartati (Solo in Anagrafica, no prezzi): {len(only_in_anagrafica)}")
print(f"ISIN scartati (Solo in Storico, no anagrafica): {len(only_in_storico)}")

In [ ]:
# Il filtraggio vero e proprio
# La funzione .isin() di Pandas è ottimizzata per questo task.
# Controlla se il valore nella colonna 'isincode' è presente nella nostra lista di ISIN comuni.
df_anagrafica_filtrato = df_anagrafica[df_anagrafica['isincode'].isin(common_isins)].copy()
df_storico_filtrato = df_storico[df_storico['isincode'].isin(common_isins)].copy()


In [ ]:
# Carichiamo solo ISIN e Data per fare velocissimi
# parse_dates e dayfirst=True sono fondamentali per fargli capire che è GG/MM/AAAA
df_dates = pd.read_csv(file_path_storico, 
                       sep=';', 
                       usecols=['isincode', 'referencedate'],
                       parse_dates=['referencedate'], 
                       dayfirst=True)

# 1. Estrazione del Time Span Globale
min_date = df_dates['referencedate'].min()
max_date = df_dates['referencedate'].max()

print(f"\n--- TIME SPAN DEL DATASET STORICO ---")
print(f"Data assoluta più vecchia: {min_date.strftime('%d/%m/%Y')}")
print(f"Data assoluta più recente: {max_date.strftime('%d/%m/%Y')}")
print(f"Profondità totale in giorni: {(max_date - min_date).days} giorni")

# 2. Analisi profonda (Quanti dati ha davvero ogni obbligazione?)
# Raggruppiamo per ISIN e calcoliamo quanti giorni di dati ha ciascuno
history_lengths = df_dates.groupby('isincode')['referencedate'].count()

print(f"\n--- ANALISI STORICO PER SINGOLO TITOLO ---")
print(f"Giorni di storico MEDIO per bond: {history_lengths.mean():.0f}")
print(f"Giorni di storico MEDIANO: {history_lengths.median():.0f}")
print(f"Titoli con meno di 6 mesi di dati (< 130 giorni borsa): {(history_lengths < 130).sum()}")
print(f"Titoli con più di 1 anno di dati (> 250 giorni borsa): {(history_lengths > 250).sum()}")

A questo punto mi sono ricordato che probabilmente tra queste obbligazioni ce ne sono alcune non standard e quindi dobbiamo escludere:
1.  **Obbligazioni non governative e non europee:** Aziendali (corporate), bancarie, ecc. hanno un rischio di credito diverso che "sporcherebbe" il modello.
2.  **Obbligazioni "esotiche":** Quelle con cedole variabili (step-up, inflation-linked come i BTP Italia/Futura) hanno un modello di prezzo completamente diverso e non possono essere confrontate con i bond a tasso fisso.

In [ ]:
# --- FASE 1.5: FILTRAGGIO AVANZATO DELL'UNIVERSO DI TRADING ---

# Lavoriamo sul DataFrame 'df_anagrafica_filtrato' che contiene già i 1309 ISIN comuni
print(f"ISIN totali prima del filtraggio avanzato: {df_anagrafica_filtrato['isincode'].nunique()}")

# 1. Filtro per Obbligazioni Governative Europee
# Usiamo il codice ISIN che inizia con il prefisso del paese (IT, DE, FR, etc.)
# e la colonna 'issuer' che di solito contiene 'GOV' per i governativi.
european_gov_codes = ['IT', 'DE', 'FR', 'ES', 'PT', 'AT', 'BE', 'NL', 'FI', 'IE', 'GR']

# Creiamo una maschera booleana: ISIN deve iniziare con uno dei codici E l'emittente deve essere governativo
is_european_gov = (df_anagrafica_filtrato['isincode'].str.startswith(tuple(european_gov_codes))) & \
                  (df_anagrafica_filtrato['description'].str.contains('GOV|Btp|Schatz|OAT|Bonos|Obligaciones|Bund|BOT|CTZ|Bubill|Zc', case=False, na=False))

df_anagrafica_gov = df_anagrafica_filtrato[is_european_gov].copy()
print(f"ISIN rimasti dopo filtro 'EU Government': {df_anagrafica_gov['isincode'].nunique()}")

# 2. Filtro per escludere Obbligazioni "Esotiche" (a cedola non fissa)
# Cerchiamo parole chiave nella descrizione del titolo.
parole_chiave_escluse = ['Futura', 'Italia', 'Inflazione', 'Indicizzato', 'Inflation', 'Valore', 'Btpi', 'Linker', "Piu"]

pattern_esclusione = '|'.join(parole_chiave_escluse)

# Creiamo una maschera per trovare le righe che contengono queste parole (case=False per ignorare maiuscole/minuscole)
is_exotic = df_anagrafica_gov['description'].str.contains(pattern_esclusione, case=False, na=False)

# Usiamo l'operatore tilde (~) per invertire la maschera, ovvero "TENIAMI TUTTO QUELLO CHE NON È ESOTICO"
df_anagrafica_clean = df_anagrafica_gov[~is_exotic].copy()
print(f"ISIN rimasti dopo filtro 'Esotici': {df_anagrafica_clean['isincode'].nunique()}")

# 3. Otteniamo la lista finale di ISIN "puri" su cui lavorare
final_clean_isins = set(df_anagrafica_clean['isincode'].unique())

print("\n--- RISULTATO FINALE ---")
print(f"Numero di obbligazioni 'Plain Vanilla' selezionate per il modello: {len(final_clean_isins)}")

# ORA POSSIAMO FILTRARE LO STORICO CON LA NUOVA LISTA DI ISIN PULITI
df_storico_clean = df_storico_filtrato[df_storico_filtrato['isincode'].isin(final_clean_isins)].copy()

print(f"Dimensioni Storico finale (pulito): {df_storico_clean.shape}")


# Fase 2: Feature Engeneering

In [ ]:
# --- CONTROLLO QUALITÀ DATI (Best Practice) ---
# 1. Duplicati su ISIN+Data
if 'isincode' in df_storico_filtrato.columns and 'referencedate' in df_storico_filtrato.columns:
    n_dupes = df_storico_filtrato.duplicated(subset=['isincode', 'referencedate']).sum()
    print(f"Duplicati su ISIN+Data nello storico filtrato: {n_dupes}")
    
# 2. Missing nelle colonne chiave
print("Missing values nelle colonne chiave dello storico:")
print(df_storico_filtrato[['isincode', 'referencedate']].isnull().sum())

if 'isincode' in df_anagrafica_filtrato.columns:
    print("Missing values nelle colonne chiave dell'anagrafica:")
    print(df_anagrafica_filtrato['isincode'].isnull().sum())

In [ ]:
# --- ESTRAZIONE DEL COUPON ---
import re

def extract_coupon(description):
    if pd.isnull(description):
        return np.nan
    desc = str(description)

    # GESTIONE ZERO COUPON: Se è un BOT o contiene "zc" (zero coupon), la cedola è 0% matematico
    if 'bot' in desc or 'Zc' in desc or 'zero' in desc or 'ctz' in desc:
        return 0.0

    # Cerchiamo pattern di coupon: numeri seguiti da % (es. "5%", "3,5%")
    match = re.search(r'(\d+[\.,]\d+|\d+)[ ]*%', desc)
    if match:
        return float(match.group(1).replace(',', '.'))
    # Cerchiamo pattern di coupon: numeri preceduti da EUR
    match = re.search(r'[Ee][Uu][Rr][ ]*(\d+[\.,]\d+|\d+)', desc)
    if match:
        return float(match.group(1).replace(',', '.'))   
    return np.nan

df_anagrafica_clean['coupon'] = df_anagrafica_clean['description'].apply(extract_coupon)
# Verfichiamo i risultati dell'estrazione del coupon
print(df_anagrafica_clean[['description', 'coupon']].head(10))
print(df_anagrafica_clean['coupon'].describe())
perc_nan = df_anagrafica_clean['coupon'].isnull().mean() * 100
print(f"Percentuale di coupon NaN: {perc_nan:.2f}%")
missing_coupon = df_anagrafica_clean[df_anagrafica_clean['coupon'].isnull()]
print(f"Obbligazioni con coupon non estratto: {missing_coupon.shape[0]}")
display(missing_coupon[['description']].head(10))

# scarto l'unico che di cui non riesco ad estrarre il coupon (è un titolo "esotico" che è sfuggito al filtro precedente)
df_anagrafica_clean = df_anagrafica_clean.dropna(subset=['coupon']).copy()

In [ ]:
# --- CALCOLO DAYS TO MATURITY (DTM) ---
# Uniamo le informazioni fisse (cedola, scadenza) al file dei prezzi giornalieri
df_ml = pd.merge(df_storico_clean, 
                 df_anagrafica_clean[['isincode', 'description', 'redemptiondate', 'coupon']], 
                 on='isincode', 
                 how='inner')

# DTM = Differenza in giorni tra la Data di Scadenza e la Data in cui è stato registrato il Prezzo
df_ml['redemptiondate'] = pd.to_datetime(df_ml['redemptiondate'], dayfirst=True, errors='coerce')
df_ml['days_to_maturity'] = (df_ml['redemptiondate'] - df_ml['referencedate']).dt.days

# A volte alcuni provider mantengono i prezzi anche dopo la scadenza, vogliamo evitare di avere dati "sporchi" con DTM negativi
# Scartiamo le righe dove il bond è già scaduto (DTM <= 0)
df_ml = df_ml[df_ml['days_to_maturity'] > 0].copy()

# Aggiungiamo anche gli ANNI alla scadenza (Years to Maturity), molto usati in finanza
df_ml['years_to_maturity'] = df_ml['days_to_maturity'] / 365.25

print(df_ml[['isincode', 'referencedate', 'pricevalue', 'coupon', 'years_to_maturity']].head(10))

In [ ]:
import pandas_datareader.data as web
import datetime

In [ ]:
# 1. Definiamo le date del tuo progetto (Dal 2022 fino alla data massima del tuo CSV)
start_date = datetime.datetime(2022, 12, 1)
end_date = datetime.datetime(2026, 5, 8)


In [ ]:

# 2. Definiamo i Ticker della FRED
macro_tickers = {
    'BCE_Rates' : 'ECBDFR',              # Tasso BCE (Giornaliero)
    'FED_Rate': 'EFFR',                 # Tasso FED (Giornaliero)
    'EU_Inflation': 'FPCPITOTLZGEMU' # Inflazione EU (Mensile)
}

print("Scaricamento dati macroeconomici da FRED in corso...")
# pandas_datareader fa una chiamata API e ci restituisce direttamente un DataFrame!
df_macro = web.DataReader(list(macro_tickers.values()), 'fred', start_date, end_date)

# Rinominiamo le colonne per renderle leggibili
df_macro.columns = list(macro_tickers.keys())


In [ ]:
# --- LA GESTIONE DEI DATI QUANTITATIVA ---

# PROBLEMA 1: L'inflazione è mensile, i tassi sono giornalieri. 
# Avremo un sacco di "NaN" (vuoti) nei giorni in cui l'inflazione non viene pubblicata.
# SOLUZIONE: "Forward Fill" (ffill). Copia il dato del mese scorso su tutti i giorni successivi
df_macro['EU_Inflation'] = df_macro['EU_Inflation'].ffill()

# PROBLEMA 2: Il "Lookahead Bias" (Fuga di dati dal futuro).
# L'inflazione di Marzo viene pubblicata il 15 Aprile. Se non "shiftiamo" il dato, 
# diremmo alla Rete Neurale il tasso di inflazione prima che il mercato lo sappia.
# SOLUZIONE: Shift (ritardo) dell'inflazione di 30 giorni!
df_macro['EU_Inflation_Lagged'] = df_macro['EU_Inflation'].shift(30)

# Scartiamo la colonna originale e teniamo solo quella ritardata, pulendo i primi 30 giorni vuoti
df_macro = df_macro.drop(columns=['EU_Inflation']).dropna()

In [ ]:

print("\n--- DATI MACROECONOMICI PRONTI ---")
display(df_macro.head())
display(df_macro.tail())

In [ ]:
df_macro_reset = df_macro.reset_index()  # Converte l'indice in colonna 'DATE'
df_macro_reset.columns = df_macro_reset.columns.str.lower()  # 'DATE' → 'date'

# Assicurati che entrambe le date siano datetime
df_ml['referencedate'] = pd.to_datetime(df_ml['referencedate'])
df_macro_reset['date'] = pd.to_datetime(df_macro_reset['date'])

# Merge con il dataset dei prezzi (df_ml) usando la data come chiave
# Metodo: left join per mantenere tutte le righe di df_ml e aggiungere i dati macro corrispondenti
df_ml_macro = pd.merge(df_ml, df_macro_reset, left_on='referencedate', right_on='date', how='left')
print("\n--- DATASET FINALE CON CARATTERISTICHE MACROAGGREGATE ---")
display(df_ml_macro.head())
display(df_ml_macro.tail())